# R4Det × V2X-Radar-V Google Colab 训练与评测手册

本 Notebook 适配 Google Colab（T4 16 GB / L4 24 GB / A100 40 GB）：
- **代码**：从 GitHub 拉取 `feat/colab-v2x-training` 分支。
- **数据（23.9 GB）**：直接高速下载到 Colab 本地 NVMe `/content/`（不占 Drive 空间）。
- **权重与 Checkpoint**：持久化到 Google Drive，断线重连后自动从 `latest.pth` 续训。
- **环境**：通过 `tools/colab_setup_env.sh` 建立与本地完全一致的 `Python 3.8 + PyTorch 2.0.1+cu118 + MMCV 1.7.2` 环境并按当前 GPU 算力编译算子。

## 1. 挂载 Google Drive 并检查 GPU / 磁盘空间
请确保菜单 `代码执行程序 → 更改运行时类型` 已选择 **GPU**。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!df -h /content

## 2. 填写仓库地址与 Google Drive 路径

In [ ]:
import os

# 代码来源：优先自动解压 Drive 里的 R4Det_code.zip（免 GitHub 配置），不存在时再 git clone
os.environ['CODE_ZIP'] = '/content/drive/MyDrive/R4Det_V2X/R4Det_code.zip'
os.environ['REPO_URL'] = 'https://github.com/SnailKorchagin/R4Det.git'
os.environ['REPO_BRANCH'] = 'feat/colab-v2x-training'

# Google Drive 持久化目录（v2：从 v1 epoch_12.pth 热启动，修正 V2X Anchor Z/尺寸与 NMS 后精调 6 轮）
os.environ['DRIVE_BASE'] = '/content/drive/MyDrive/R4Det_V2X'
os.environ['INIT_CKPT'] = '/content/drive/MyDrive/R4Det_V2X/stage4_full_single_frame/epoch_12.pth'
os.environ['WORK_DIR'] = '/content/drive/MyDrive/R4Det_V2X/stage4_v2_anchor_fix'
os.environ['EVAL_DIR'] = '/content/drive/MyDrive/R4Det_V2X/stage5_eval_v2'
os.environ['MAX_EPOCHS'] = '6'
os.environ['LR'] = '1e-4'

print('Config set. WORK_DIR =', os.environ['WORK_DIR'])

## 3. 拉取代码并一键配置 Python 3.8 + CUDA 11.8 + MMCV 1.7.2 环境

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ ! -d /content/R4Det ]; then
  if [ -f "$CODE_ZIP" ]; then
    echo "Extracting $CODE_ZIP -> /content/R4Det ..."
    unzip -q -o "$CODE_ZIP" -d /content/
  else
    git clone -b "$REPO_BRANCH" "$REPO_URL" /content/R4Det
  fi
else
  echo "Reusing existing /content/R4Det"
  if [ -d /content/R4Det/.git ]; then
    git -C /content/R4Det pull --ff-only origin "$REPO_BRANCH" || true
  fi
fi
cd /content/R4Det
bash tools/colab_setup_env.sh /content/r4det_env

## 4. 高速下载全量 V2X-Radar-V（11,390 帧）并生成 `.pkl`
解压后自动删除 `.zip` 以释放 23 GB 磁盘空间。

In [ ]:
%%bash
set -euo pipefail
cd /content/R4Det
mkdir -p .downloads/V2X-Radar-V data/V2X-Radar-V checkpoints

mkdir -p "$(dirname "$INIT_CKPT")"
if [ ! -f "$INIT_CKPT" ]; then
  echo "Downloading official final_tj4d.pth from HuggingFace to Google Drive..."
  curl -L -C - -o "$INIT_CKPT" \
    "https://huggingface.co/Hoeyy/R4Det/resolve/main/final_tj4d.pth"
fi
ln -sf "$INIT_CKPT" checkpoints/final_tj4d.pth
echo "Linked INIT_CKPT -> checkpoints/final_tj4d.pth"

if [ ! -f data/V2X-Radar-V/v2x_radar_v_infos_train.pkl ]; then
  curl -L -C - -o .downloads/V2X-Radar-V/ImageSets.zip \
    "https://huggingface.co/datasets/yanglei18/V2X-Radar/resolve/main/V2X-Radar-V/ImageSets.zip"
  curl -L -C - -o .downloads/V2X-Radar-V/V2X-Radar-V.zip \
    "https://huggingface.co/datasets/yanglei18/V2X-Radar/resolve/main/V2X-Radar-V/V2X-Radar-V.zip"

  unzip -q -o .downloads/V2X-Radar-V/V2X-Radar-V.zip -d data/
  unzip -q -o .downloads/V2X-Radar-V/ImageSets.zip -d data/V2X-Radar-V/
  rm -f .downloads/V2X-Radar-V/V2X-Radar-V.zip

  /content/r4det_env/bin/python tools/data_converter/v2x_radar_v_converter.py \
    --data-root data/V2X-Radar-V \
    --splits train val trainval test \
    --class-scheme tj4d_compat
else
  echo "Dataset .pkl already prepared under data/V2X-Radar-V/."
fi

# 快速运行阶段 1 几何检查
/content/r4det_env/bin/python tools/inspect_v2x_radar_v.py \
  --data-root data/V2X-Radar-V \
  --frame-id 000009 \
  --out-dir "$DRIVE_BASE/inspection"

## 5. 阶段 4：全量单帧训练（自动适配显存 & 自动断点续训）
- 自动检测 GPU 显存：`A100` (`batch=4`) / `L4` (`batch=2, accum=2`) / `T4` (`batch=2, accum=2, freeze depth_net`)。
- 每 **250 个更新步** 或每 **30 分钟**（先到为准）自动原子保存 `latest.pth` 到 Google Drive，断线重连后精确从上次中断的 step 续训，不再白跑整轮。

In [ ]:
%%bash
cd /content/R4Det
bash tools/colab_train_v2x.sh

## 6. 阶段 5：全量验证集（1,498 帧）评测 + 训练曲线 + 本地逐帧展示页

In [ ]:
%%bash
set -euo pipefail
cd /content/R4Det
export PATH="/content/r4det_env/bin:/usr/bin:/bin"
export CUDA_HOME="/content/r4det_env"
export CC="/content/r4det_env/bin/x86_64-conda-linux-gnu-cc"
export CXX="/content/r4det_env/bin/x86_64-conda-linux-gnu-c++"
export TORCH_CUDA_ARCH_LIST="$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -n 1 | tr -d '[:space:]')"
export TORCH_EXTENSIONS_DIR="/content/R4Det/.cache/torch_extensions"
export MPLCONFIGDIR="/tmp/r4det-colab-mpl"

/content/r4det_env/bin/python -u tools/eval_v2x_radar_v.py \
  --config configs/r4det/V2X-Radar-V-R4Det_single_frame.py \
  --checkpoint "$WORK_DIR/latest.pth" \
  --ann-file data/V2X-Radar-V/v2x_radar_v_infos_val.pkl \
  --out-dir "$EVAL_DIR" \
  --train-metrics "$WORK_DIR/train_metrics.jsonl" \
  --max-vis-frames 30